In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer 
import torch
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from transformers import AutoModelForCausalLM
import json
from sklearn.model_selection import train_test_split as tts
from peft import LoraConfig
from peft import get_peft_model

In [2]:
# Importing LoRa 
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [3]:
# cuda verification
device=torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
# tokenizer is getting fetched 
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)

In [6]:
with open("dataset.json","r",encoding="utf-8") as f:
    text_data=json.load(f)

text_data[0]

{'user': 'Develop a Python program snippet to Determine High Sneezing and coughing etiquette: Preventing Spread of Germs for Engineer for Experts. Incorporate if/else or switch/case statements to handle various cases related to the Bias. Dry-run, ensure your control flow logic is clear and well-commented.',
 'content': 'Here is python code which determines high sneezing and coughing etiquette based upon different conditions :\n\n\n```python \ndef determine_high_sneeze_cough_etiquette(isolation_period):\n    # check whether isolation period has been exceeded \n    if (isolation_period >0) & (today() - last_symptomatic_day() < isolation_period):\n        return "Stay home"\n    \n    elif exposure_risk() >= risk_threshold():\n        return "Wear mask"\n        \n    else: \n        return "Go about usual activities"\n    \n        \n# define function to calculate days since symptoms appeared    \ndef today():\n    return datetime.now().date()\n    \n    \n# define function to retrieve d

In [7]:
train_data,test_data=tts(text_data,
                         test_size=0.2,
                         random_state=42)

train_data[0]

{'user': 'Create a Ruby script snippet that Determines Low Hygiene during illness: Personal Hygiene for Better Recovery for Decision Making for Beginners. Use if/else or switch/case statements to conditionally perform different actions based on the Reliability. Dry-run, then include comments that outline the control flow and how you handle different scenarios.',
 'content': 'Here is a possible implementation of such a script written in Ruby:\n```ruby \n# Function to determine low hygiene level based on personal health symptoms\ndef check_hygiene(symptoms)\n  # Define conditions for each severity level of hygiene\n  case \n  when (symptoms[\'cough\'] && !symptoms[\'fever\']) \n    || (!symptoms[\'cough\'] && symptoms[\'soreThroat\'])\n    return \'Mild\'\n  when symptoms[\'diarrhea\']\n    return \'Moderate\'\n  else \n    return \'Severe\'\n  end\nend\n\n# Example usage of function to determine hygiene level\nsymptoms = { cough: true, fever: false, soreThroat: true, diarrhea: false }\n

In [8]:
class CustomDataset(Dataset):
    def __init__(self,text):
        self.text=text
    
    def __len__(self):
        return len(self.text)
    
    def __getitem__(self, idx):
        sample = self.text[idx]

        messages = [
            {"role": "user", "content": sample['user']},
            {"role": "assistant", "content": sample["content"]}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

In [9]:
# Getting a tensor 
train_dataset=CustomDataset(train_data)
test_dataset=CustomDataset(test_data)

In [10]:
# Data is Loaded into Batches now 
train_loader=DataLoader(train_dataset,batch_size=1,shuffle=True,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=1,shuffle=True,pin_memory=True)


In [11]:
# Getting the model 
model=AutoModelForCausalLM.from_pretrained( "Qwen/Qwen2.5-0.5B-Instruct")
model.to(device)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [12]:
# Attaching LoRa with the Model 
model = get_peft_model(
    model,
    lora_config
)


W0710 13:45:21.384000 15708 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [13]:
print(len(tokenizer))
print(model.get_input_embeddings().weight.shape[0])

151665
151936


In [14]:
#Important Parameters
epochs=2
learning_rate = 2e-4

In [15]:
# Optimizer Upload 
optimizer=optim.Adam(model.parameters(),lr=learning_rate)

In [16]:
# Training the Model
for epoch in range(epochs):
    for batch in train_loader:
        batch={k:v.to(device) for k,v in batch.items()}
        # forward pass

        outputs=model(input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["input_ids"])

        #Loss Function
        loss=outputs.loss

        #Accumulation Problem 
        optimizer.zero_grad()

        # backward
        loss.backward()

        #Optimization
        optimizer.step()
    print("Loss:", loss.item())
    print(
        torch.isnan(outputs.logits).any(),
        torch.isinf(outputs.logits).any()
    )

Loss: 0.6430705785751343
tensor(False, device='cuda:0') tensor(False, device='cuda:0')
Loss: 1.04914391040802
tensor(False, device='cuda:0') tensor(False, device='cuda:0')


In [ ]:
model_merged = AutoModelForCausalLM.from_pretrained(
    "./Qwen_Merged",
    torch_dtype=torch.float16
)
model_merged.to(device)

tokenizer_merged = AutoTokenizer.from_pretrained("./Qwen_Merged")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [17]:
def generate_text(prompt, model, tokenizer, device, max_new_tokens=512):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

In [23]:
prompt = """Instruction:
Write a Python function to reverse a string.

Response:
"""
generate_text(prompt,model_merged,tokenizer_merged,device)

'Instruction:\nWrite a Python function to reverse a string.\n\nResponse:\n```python\ndef reverse_string(s):\n    return s[::-1]\n\n# Test the function with provided data points\nprint(reverse_string("hello"))  # Output: "olleh"\nprint(reverse_string("world"))  # Output: "dlrow"\n```\n\nNote:\n- The `::-1` slicing syntax is used to reverse the order of elements in a sequence.\n- The reversed string is then returned as the output.'

In [19]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained("./Qwen_Merged")
tokenizer.save_pretrained("./Qwen_Merged")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./Qwen_Merged\\tokenizer_config.json',
 './Qwen_Merged\\chat_template.jinja',
 './Qwen_Merged\\tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

path = r"C:\Users\Daksh\Desktop\DRDO_Project\Qwen_Merged"

tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(path)

print("Model Loaded Successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model Loaded Successfully!
